# 07. MuJoCo Playground & MJX: 대규모 병렬 시뮬레이션과 로봇 강화학습 (Locomotion & Dexterous Manipulation)

![MuJoCo Playground Banner](../assets/imgs/playground_banner.png)

이 노트북에서는 Google DeepMind의 **[MuJoCo Playground](https://github.com/google-deepmind/mujoco_playground)**와 **MuJoCo MJX (`mujoco.mjx`)**의 핵심 아키텍처 및 로봇 학습 기법을 Windows 환경에서 단계별로 실습합니다.

---

### 학습 목표
1. **MuJoCo MJX (`mujoco.mjx`)와 `jax.vmap` 대규모 병렬 물리 시뮬레이션**:
   - `mjModel` / `mjData`를 JAX 배열 기반의 `mjx.Model` / `mjx.Data`로 변환하고, `jax.jit` 및 `jax.vmap`을 통해 **수백~수천 개의 로봇 환경을 동시에 시뮬레이션**하는 원리와 처리량(Steps/sec)을 분석합니다.
2. **Playground 환경 설계 패턴 (`sim_dt` vs `ctrl_dt`, `reward.tolerance`) & 병렬 MPPI 제어**:
   - 물리 시뮬레이션 주기(`sim_dt`)와 에이전트 제어 주기(`ctrl_dt`, 서브스테핑) 분리 구조, DeepMind Control Suite의 표준 보상 함수(`tolerance`), 그리고 MJX 병렬 롤아웃을 활용한 **실시간 샘플링 기반 최적 제어(MPPI)**를 구현합니다.
3. **사족보행(Quadruped Locomotion — Unitree Go1) 조이스틱 추종, 외란, Sim-to-Real 도메인 랜덤화**:
   - `assets/mujoco_menagerie/unitree_go1` 모델을 활용하여 조이스틱 속도 명령($v_x, v_y, \omega_{\text{yaw}}$) 3D 화살표 시각화, 발 끝 스윙 높이(`swing_peak`) 진단 플롯, 외란(Velocity Kick) 반응, 그리고 **Sim-to-Real 전이를 위한 도메인 랜덤화(Domain Randomization)**를 실습합니다.
4. **다지 로봇 손 조작(Dexterous Manipulation — LEAP Hand $SO(3)$ 큐브 회전) & 비대칭 액터-크리틱**:
   - `assets/mujoco_menagerie/leap_hand` 모델과 3D 큐브를 결합하여 **인핸드 큐브 방향 제어(In-Hand Reorientation)** 환경을 구성하고, 실제 센서 관측(`state`)과 시뮬레이터 특권 관측(`privileged_state`)을 분리하는 **비대칭 액터-크리틱(Asymmetric Actor-Critic)** 구조를 학습합니다.
5. **`mujoco_playground` & `brax` PPO 전체 학습 및 커리큘럼 파인튜닝 파이프라인**:
   - 우리 프로젝트의 `assets/mujoco_menagerie` 서브모듈을 `mujoco_playground`에 직접 연결하고, Brax PPO 학습 및 에너지 페널티(`energy`, `dof_acc`) 파인튜닝 코드를 살펴봅니다.

In [ ]:
#@title 1. Windows 노트북 환경 초기화 및 필수 패키지 임포트
import functools
import time
from typing import Dict, Tuple

import jax
import jax.numpy as jp
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import numpy as np

from utils.notebook_setup import (
    MENAGERIE_DIR,
    get_robot_scene_path,
    init_notebook,
    list_menagerie_robots,
)

project_root = init_notebook()
np.set_printoptions(precision=3, suppress=True, linewidth=100)

print(f"MuJoCo 버전: {mujoco.__version__}")
print(f"JAX 버전: {jax.__version__} (기본 백엔드: {jax.default_backend()})")
print(f"Menagerie 서브모듈 내 사용 가능 로봇 수: {len(list_menagerie_robots())}종")

---
## 파트 1. MuJoCo MJX (`mujoco.mjx`)와 `jax.vmap` 대규모 병렬 물리 시뮬레이션

기존 01~06번 노트북에서는 CPU에서 동작하는 C/C++ 엔진 바인딩(`mujoco.mj_step(model, data)`)을 사용했습니다.
반면 **MuJoCo Playground**의 기반이 되는 **MuJoCo MJX (`mujoco.mjx`)**는 물리 엔진 전체를 **JAX(XLA)** 연산 그래프로 재구현하여 다음과 같은 강력한 기능을 제공합니다:

1. **`jax.jit` 컴파일**: 물리 스텝(`mjx.step`)과 제어기·보상 함수를 하나의 XLA 커널로 융합(Fusion)하여 파이썬 오버헤드를 제거합니다.
2. **`jax.vmap` 대규모 병렬화**: 단일 환경 스텝 함수를 `jax.vmap(mjx.step, in_axes=(None, 0))`로 감싸기만 하면 $N=1,024 \sim 8,192$개의 독립적인 로봇 환경을 단일 디바이스(GPU/CPU)에서 동시에 진행할 수 있습니다.
3. **도메인 랜덤화(`in_axes=(0, 0)`)**: 모델 파라미터(`mjx.Model`의 마찰계수, 질량 등)의 배치 축까지 `in_axes`로 열어주면 각 병렬 환경마다 물리 법칙이 조금씩 다른 **Sim-to-Real 학습 환경**이 즉시 만들어집니다.

In [ ]:
# MJX 임포트 및 카트폴(Cartpole) MJCF 모델 정의
try:
    from mujoco import mjx
    HAS_MJX = True
    print("mujoco.mjx 모듈 로드 성공!")
except ImportError:
    HAS_MJX = False
    print("[안내] mujoco-mjx가 아직 설치되지 않았습니다. 'uv sync' 실행 시 자동으로 설치됩니다.")

CARTPOLE_XML = """
<mujoco model="cartpole_playground">
  <option timestep="0.01" integrator="Euler">
    <flag contact="disable"/>
  </option>
  <visual>
    <headlight diffuse="0.7 0.7 0.7" ambient="0.3 0.3 0.3"/>
  </visual>
  <worldbody>
    <light pos="0 0 2.5"/>
    <camera name="fixed" pos="0 -2.6 0.6" xyaxes="1 0 0 0 0.2 1"/>
    <geom name="rail" type="capsule" fromto="-1.8 0 0.5 1.8 0 0.5" size="0.02" rgba="0.3 0.3 0.35 1"/>
    <body name="cart" pos="0 0 0.5">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.6 1.6" damping="0.1"/>
      <geom name="cart_geom" type="box" size="0.18 0.1 0.06" rgba="0.2 0.5 0.8 1" mass="1.0"/>
      <body name="pole" pos="0 0 0">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.005"/>
        <geom name="pole_geom" type="capsule" fromto="0 0 0 0 0 0.6" size="0.035" rgba="0.9 0.35 0.2 1" mass="0.2"/>
        <site name="pole_tip" pos="0 0 0.6" size="0.05" rgba="1 0.85 0.1 1"/>
      </body>
    </body>
  </worldbody>
  <actuator>
    <motor name="slide_motor" joint="slider" gear="15" ctrlrange="-1 1" ctrllimited="true"/>
  </actuator>
</mujoco>
"""

mj_model = mujoco.MjModel.from_xml_string(CARTPOLE_XML)
mj_data = mujoco.MjData(mj_model)
print(f"자유도(nv): {mj_model.nv}, 액추에이터(nu): {mj_model.nu}, 타임스텝(dt): {mj_model.opt.timestep}s")

In [ ]:
# MJX(jax.vmap) 병렬 시뮬레이션 처리량(Steps/sec) 벤치마크
if HAS_MJX:
    mjx_model = mjx.put_model(mj_model)

    @jax.jit
    def init_batch_env(rng: jax.Array):
        qpos = jax.random.uniform(rng, (mj_model.nq,), minval=-0.2, maxval=0.2)
        d = mjx.make_data(mj_model)
        return d.replace(qpos=qpos)

    @jax.jit
    def step_batch_env(batch_data, batch_ctrl):
        def single_step(d, u):
            d = d.replace(ctrl=u)
            return mjx.step(mjx_model, d)
        return jax.vmap(single_step)(batch_data, batch_ctrl)

    batch_sizes = [1, 16, 64, 256, 1024]
    n_steps = 50
    sps_results = []

    for B in batch_sizes:
        keys = jax.random.split(jax.random.PRNGKey(0), B)
        b_data = jax.vmap(init_batch_env)(keys)
        b_ctrl = jp.zeros((B, mj_model.nu))

        # Warm-up (XLA JIT 컴파일)
        b_data = step_batch_env(b_data, b_ctrl)
        jax.block_until_ready(b_data.qpos)

        t0 = time.perf_counter()
        for _ in range(n_steps):
            b_data = step_batch_env(b_data, b_ctrl)
        jax.block_until_ready(b_data.qpos)
        elapsed = time.perf_counter() - t0
        total_steps = B * n_steps
        sps = total_steps / elapsed
        sps_results.append(sps)
        print(f"배치 크기 B={B:4d} | 총 {total_steps:6d} 스텝 | 처리 속도: {sps:,.0f} steps/sec")

    plt.figure(figsize=(7, 3.8))
    plt.plot(batch_sizes, sps_results, "o-", linewidth=2, color="royalblue")
    plt.xscale("log", base=2)
    plt.xlabel("병렬 환경 수 (Batch Size B)")
    plt.ylabel("초당 시뮬레이션 스텝 수 (Steps / sec)")
    plt.title("MJX jax.vmap 배치 크기에 따른 시뮬레이션 처리량(Throughput)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## 파트 2. Playground 환경 설계 패턴 (`sim_dt` vs `ctrl_dt`, `reward.tolerance`) & 병렬 샘플링 제어 (MPPI)

`mujoco_playground`의 모든 환경(`dm_control_suite`, `locomotion`, `manipulation`)은 다음과 같은 공통 설계 구조를 따릅니다:

1. **서브스테핑 (`sim_dt` vs `ctrl_dt`)**:
   - 물리 엔진이 안정적으로 적분하기 위한 타임스텝 `sim_dt` (예: `0.005s` ~ `0.01s`)와 강화학습 정책이 행동을 출력하는 주기 `ctrl_dt` (예: `0.02s` ~ `0.04s`)를 분리합니다.
   - 한 번의 `env.step(state, action)` 호출 내부에서 물리 스텝은 `n_substeps = int(ctrl_dt / sim_dt)`회 반복 실행됩니다.
2. **정규화된 허용 오차 보상 함수 (`reward.tolerance`)**:
   - 목표 구간 $[b_{\text{lower}}, b_{\text{upper}}]$ 안에서는 보상이 $1.0$이고, 구간 밖으로 벗어날 때 `margin` 폭에 따라 가우시안(Gaussian) 또는 코사인 형태로 부드럽게 $0.0$으로 감소하는 함수로, 희소 보상(Sparse Reward) 문제를 해결해 줍니다.

In [ ]:
def reward_tolerance(
    x: np.ndarray,
    bounds: Tuple[float, float] = (0.0, 0.0),
    margin: float = 1.0,
    value_at_margin: float = 0.1,
) -> np.ndarray:
    """MuJoCo Playground (mujoco_playground._src.reward.tolerance) 가우시안 보상 함수 구현."""
    lower, upper = bounds
    in_bounds = np.logical_and(lower <= x, x <= upper)
    if margin == 0:
        return np.where(in_bounds, 1.0, 0.0)
    d = np.where(x < lower, lower - x, x - upper) / margin
    scale = np.sqrt(-2 * np.log(value_at_margin))
    return np.where(in_bounds, 1.0, np.exp(-0.5 * (d * scale) ** 2))

# 보상 함수 형태 시각화
x_vals = np.linspace(-2.0, 2.0, 400)
plt.figure(figsize=(7, 3.5))
plt.plot(x_vals, reward_tolerance(x_vals, bounds=(-0.1, 0.1), margin=0.5), label="bounds=(-0.1, 0.1), margin=0.5", linewidth=2)
plt.plot(x_vals, reward_tolerance(x_vals, bounds=(-0.2, 0.2), margin=1.0), label="bounds=(-0.2, 0.2), margin=1.0", linewidth=2, linestyle="--")
plt.xlabel("오차 (Error x)")
plt.ylabel("보상 (Reward [0, 1])")
plt.title("MuJoCo Playground의 reward.tolerance() 보상 곡선")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 병렬 롤아웃을 활용한 실시간 MPPI (Model Predictive Path Integral) 카트폴 제어

사전 학습된 신경망 가중치가 없더라도, 매 제어 주기(`ctrl_dt`)마다 $K=64$개의 후보 행동 시퀀스를 병렬로 샘플링하고 미래 $H=15$ 스텝의 누적 보상($R_k$)을 평가한 뒤, 보상의 지수 가중 평균으로 최적 행동 $u^*$를 계산하는 **MPPI 제어기**를 실행해 봅니다:

$$u_t^* = \frac{\sum_{k=1}^{K} \exp\left(\frac{1}{\lambda} R_k\right) u_{t}^{(k)}}{\sum_{k=1}^{K} \exp\left(\frac{1}{\lambda} R_k\right)}$$

In [ ]:
# 카트폴 스윙업 & 밸런싱 MPPI 시뮬레이션 및 비디오 렌더링
def cartpole_step_reward(qpos: np.ndarray, qvel: np.ndarray, ctrl: float) -> float:
    x, theta = qpos[0], qpos[1]
    upright = (np.cos(theta) + 1.0) / 2.0
    centered = reward_tolerance(x, bounds=(-0.05, 0.05), margin=1.2)
    small_vel = reward_tolerance(qvel[1], bounds=(-0.5, 0.5), margin=4.0)
    small_ctrl = (4.0 + reward_tolerance(ctrl, bounds=(-0.1, 0.1), margin=0.9)) / 5.0
    return float(upright * centered * small_vel * small_ctrl)

# 초기 각도: 약간 기울어진 상태(theta = 0.45 rad ≈ 26도)에서 외란 극복 및 중앙 복귀
sim_model = mujoco.MjModel.from_xml_string(CARTPOLE_XML)
sim_data = mujoco.MjData(sim_model)
rollout_data = mujoco.MjData(sim_model)
renderer = mujoco.Renderer(sim_model, height=360, width=640)

sim_data.qpos[:] = [0.6, 0.45]  # 카트 위치 +0.6m, 폴 각도 +26도에서 시작
mujoco.mj_forward(sim_model, sim_data)

K_samples = 48     # 매 스텝 샘플링할 후보 궤적 수
H_horizon = 14     # 예측 호라이즌 (스텝 수)
n_substeps = 2     # ctrl_dt = 0.02s (sim_dt = 0.01s * 2)
lambda_temp = 0.15
rng_np = np.random.default_rng(42)

mean_u = np.zeros(H_horizon)
frames = []
reward_history = []
qpos_history = []

for step in range(150):
    # K개의 후보 행동 시퀀스 샘플링
    noise = rng_np.normal(0.0, 0.45, size=(K_samples, H_horizon))
    u_candidates = np.clip(mean_u[None, :] + noise, -1.0, 1.0)
    returns = np.zeros(K_samples)

    for k in range(K_samples):
        rollout_data.qpos[:] = sim_data.qpos
        rollout_data.qvel[:] = sim_data.qvel
        total_r = 0.0
        for h in range(H_horizon):
            rollout_data.ctrl[0] = u_candidates[k, h]
            for _ in range(n_substeps):
                mujoco.mj_step(sim_model, rollout_data)
            total_r += (0.95 ** h) * cartpole_step_reward(rollout_data.qpos, rollout_data.qvel, u_candidates[k, h])
        returns[k] = total_r

    # 지수 가중 평균(Softmax)으로 최적 제어 입력 계산
    weights = np.exp((returns - np.max(returns)) / lambda_temp)
    weights /= np.sum(weights)
    optimal_seq = np.sum(weights[:, None] * u_candidates, axis=0)
    u_star = optimal_seq[0]

    # Warm-start: 다음 스텝을 위해 평균 시퀀스 시프트
    mean_u[:-1] = optimal_seq[1:]
    mean_u[-1] = 0.0

    # 실제 환경에 u_star 인가 (n_substeps)
    sim_data.ctrl[0] = u_star
    for _ in range(n_substeps):
        mujoco.mj_step(sim_model, sim_data)

    r_now = cartpole_step_reward(sim_data.qpos, sim_data.qvel, u_star)
    reward_history.append(r_now)
    qpos_history.append(sim_data.qpos.copy())

    renderer.update_scene(sim_data, camera="fixed")
    frames.append(renderer.render())

renderer.close()
media.show_video(frames, fps=50)

qpos_arr = np.array(qpos_history)
t_axis = np.arange(len(reward_history)) * (sim_model.opt.timestep * n_substeps)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(t_axis, qpos_arr[:, 0], label="카트 위치 x (m)")
axes[0].plot(t_axis, qpos_arr[:, 1], label="폴 각도 θ (rad)")
axes[0].axhline(0.0, color="k", linestyle="--", alpha=0.5)
axes[0].set_xlabel("시간 (s)")
axes[0].set_title("카트 위치 및 폴 각도 수렴 과정")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_axis, reward_history, color="forestgreen", linewidth=2)
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("시간 (s)")
axes[1].set_ylabel("Step Reward")
axes[1].set_title("Playground reward.tolerance 기반 보상 추이")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 파트 3. 사족보행(Quadruped Locomotion — Unitree Go1) 조이스틱 추종, 외란 및 도메인 랜덤화

`mujoco_playground`의 `locomotion.ipynb`에서 다루는 핵심 주제는 **Unitree Go1 / Go2 4족 보행 로봇**의 조이스틱 속도 명령 추종(`Go1JoystickFlatTerrain`)과 Sim-to-Real 전이 기법입니다.

여기서는 방금 서브모듈로 연결한 **`assets/mujoco_menagerie/unitree_go1/scene.xml`**을 직접 불러와 다음 4가지를 실습합니다:
1. **조이스틱 명령($\mathbf{v}_{\text{cmd}} = [v_x, v_y, \omega_{\text{yaw}}]^T$) 및 상단 3D 화살표 시각화**: MuJoCo 렌더러의 `scn.geoms`에 커스텀 화살표 지오메트리(`mjGEOM_ARROW`)를 동적으로 추가하여 로봇이 목표로 하는 이동 방향과 속도를 실시간 표시합니다.
2. **트로트(Trot) 보행 패턴 & 외란(Velocity Kick Perturbation) 인가**: 대각선 다리 쌍(FR+RL vs FL+RR)이 교차 스윙하는 트로트 보행 제어기를 구동하고, 주행 도중 측면 속도 충격(Velocity Kick)이 가해졌을 때의 균형 회복을 시뮬레이션합니다.
3. **보행 진단 플롯 (Playground Diagnostic Plots)**: 4개 발 끝(FR, FL, RR, RL)의 수직 스윙 높이 궤적(`swing_peak` $2 \times 2$ 그리드)과 명령 속도 대비 실제 선속도/각속도 추종 성능을 시각화합니다.
4. **Sim-to-Real 도메인 랜덤화 (Domain Randomization)**: `jax.vmap`을 활용해 바닥 마찰계수(`geom_friction`), 관절 전기자 관성(`dof_armature`), 몸통 질량 중심(`body_ipos`)을 병렬 환경마다 무작위화하는 방법을 구현합니다.

In [ ]:
# 1. Menagerie 서브모듈에서 Unitree Go1 모델 로드 및 구조 확인
go1_xml_path = get_robot_scene_path("unitree_go1")
go1_model = mujoco.MjModel.from_xml_path(go1_xml_path)
go1_data = mujoco.MjData(go1_model)

# 'home' 키프레임으로 초기화
home_key_id = mujoco.mj_name2id(go1_model, mujoco.mjtObj.mjOBJ_KEY, "home")
mujoco.mj_resetDataKeyframe(go1_model, go1_data, home_key_id)
mujoco.mj_forward(go1_model, go1_data)

print(f" 로드된 모델 경로: {go1_xml_path}")
print(f" - 일반화 좌표 수 (nq): {go1_model.nq} (7 Freejoint + 12 관절)")
print(f" - 속도 자유도 수 (nv): {go1_model.nv} (6 Base + 12 관절)")
print(f" - 액추에이터 수 (nu): {go1_model.nu}")
print(" - 액추에이터 목록:", [mujoco.mj_id2name(go1_model, mujoco.mjtObj.mjOBJ_ACTUATOR, i) for i in range(go1_model.nu)])

In [ ]:
# 2. 조이스틱 명령 3D 화살표 렌더링 헬퍼 & 트로트(Trot) 보행 + 측면 외란(Velocity Kick) 시뮬레이션
def add_joystick_arrow(scn: mujoco.MjvScene, torso_pos: np.ndarray, yaw: float, cmd_vx: float, cmd_vy: float):
    """Playground의 draw_joystick_command와 동일하게 로봇 몸통 위에 목표 속도 벡터 화살표를 추가합니다."""
    if scn.ngeom >= scn.maxgeom:
        return
    arrow_start = torso_pos + np.array([0.0, 0.0, 0.22])
    # 몸통 좌표계 명령 속도를 월드 좌표계로 회전
    cos_y, sin_y = np.cos(yaw), np.sin(yaw)
    world_vx = cos_y * cmd_vx - sin_y * cmd_vy
    world_vy = sin_y * cmd_vx + cos_y * cmd_vy
    arrow_end = arrow_start + np.array([world_vx * 0.45, world_vy * 0.45, 0.0])

    geom = scn.geoms[scn.ngeom]
    mujoco.mjv_initGeom(
        geom,
        type=mujoco.mjtGeom.mjGEOM_ARROW,
        size=np.zeros(3),
        pos=np.zeros(3),
        mat=np.eye(3).flatten(),
        rgba=np.array([0.1, 0.9, 0.3, 0.85], dtype=np.float32),
    )
    mujoco.mjv_connector(geom, mujoco.mjtGeom.mjGEOM_ARROW, 0.018, arrow_start, arrow_end)
    scn.ngeom += 1

# Go1 4개 발(FR, FL, RR, RL) 지오메트리/바디 ID 조회
foot_body_names = ["FR_calf", "FL_calf", "RR_calf", "RL_calf"]
foot_body_ids = [mujoco.mj_name2id(go1_model, mujoco.mjtObj.mjOBJ_BODY, name) for name in foot_body_names]
trunk_id = mujoco.mj_name2id(go1_model, mujoco.mjtObj.mjOBJ_BODY, "trunk")

mujoco.mj_resetDataKeyframe(go1_model, go1_data, home_key_id)
mujoco.mj_forward(go1_model, go1_data)
home_ctrl = go1_data.ctrl.copy()

go1_renderer = mujoco.Renderer(go1_model, height=360, width=640)
scene_opt = mujoco.MjvOption()
scene_opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True

cam = mujoco.MjvCamera()
mujoco.mjv_defaultCamera(cam)
cam.type = mujoco.mjtCamera.mjCAMERA_TRACKING
cam.trackbodyid = trunk_id
cam.distance = 1.65
cam.elevation = -20
cam.azimuth = 135

go1_frames = []
foot_heights = []
linvel_history = []
cmd_history = []

sim_duration = 4.0
fps = 50
gait_freq = 2.8  # 2.8 Hz 트로트 보행 주파수

while go1_data.time < sim_duration:
    t = go1_data.time
    # 시간에 따라 전진 속도 명령(vx)을 0.3 m/s -> 0.6 m/s로 점진적 증가
    cmd_vx = 0.35 if t < 2.0 else 0.65
    cmd_vy = 0.0

    # t = 2.2초 부근에서 측면 외란(Velocity Kick Perturbation) 인가!
    if abs(t - 2.2) < go1_model.opt.timestep * 0.5:
        go1_data.qvel[1] += 0.45  # 측면(+y) 속도 충격 인가

    # 대각선 다리 쌍 위상차 π (FR+RL: phase 0, FL+RR: phase π)
    phase_a = 2 * np.pi * gait_freq * t
    phase_b = phase_a + np.pi

    swing_a = np.maximum(0.0, np.sin(phase_a))
    swing_b = np.maximum(0.0, np.sin(phase_b))
    stride_a = np.cos(phase_a) * (0.22 * cmd_vx)
    stride_b = np.cos(phase_b) * (0.22 * cmd_vx)

    # 몸통 롤/피치 안정화 피드백
    trunk_mat = go1_data.xmat[trunk_id].reshape(3, 3)
    roll_approx = trunk_mat[2, 1]
    pitch_approx = -trunk_mat[2, 0]

    ctrl = home_ctrl.copy()
    # 액추에이터 순서: FR(0,1,2), FL(3,4,5), RR(6,7,8), RL(9,10,11) -> [hip, thigh, calf]
    for leg_idx, (swing, stride) in enumerate([
        (swing_a, stride_a),  # FR
        (swing_b, stride_b),  # FL
        (swing_b, stride_b),  # RR
        (swing_a, stride_a),  # RL
    ]):
        base_idx = leg_idx * 3
        ctrl[base_idx + 0] += -0.35 * roll_approx
        ctrl[base_idx + 1] += stride + 0.22 * swing - 0.3 * pitch_approx
        ctrl[base_idx + 2] += -0.45 * swing

    go1_data.ctrl[:] = ctrl
    mujoco.mj_step(go1_model, go1_data)

    if len(go1_frames) < go1_data.time * fps:
        # 발 위치(z) 및 몸통 속도 기록
        f_z = [max(0.0, go1_data.xpos[bid, 2] - 0.08) for bid in foot_body_ids]
        foot_heights.append(f_z)
        linvel_history.append(go1_data.qvel[:2].copy())
        cmd_history.append([cmd_vx, cmd_vy])

        go1_renderer.update_scene(go1_data, camera=cam, scene_option=scene_opt)
        yaw = np.arctan2(trunk_mat[1, 0], trunk_mat[0, 0])
        add_joystick_arrow(go1_renderer.scene, go1_data.xpos[trunk_id], yaw, cmd_vx, cmd_vy)
        go1_frames.append(go1_renderer.render())

go1_renderer.close()
print("초록색 화살표: 조이스틱 목표 속도 명령 | 빨간색 점: 지면 접촉점(Contact Point) | t=2.2s 측면 외란 인가")
media.show_video(go1_frames, fps=fps)

In [ ]:
# 3. Playground 스타일 보행 진단 플롯: 4개 발 스윙 높이(2x2 그리드) 및 속도 명령 추종
foot_heights_arr = np.array(foot_heights)
linvel_arr = np.array(linvel_history)
cmd_arr = np.array(cmd_history)
time_axis = np.linspace(0, sim_duration, len(foot_heights_arr))

leg_labels = ["FR (우측 전방)", "FL (좌측 전방)", "RR (우측 후방)", "RL (좌측 후방)"]
leg_colors = ["crimson", "seagreen", "royalblue", "darkorange"]

fig, axs = plt.subplots(2, 2, figsize=(10, 4.5), sharex=True)
for i, ax in enumerate(axs.flat):
    ax.plot(time_axis, foot_heights_arr[:, i], color=leg_colors[i], linewidth=1.8)
    ax.axhline(0.08, color="k", linestyle="--", alpha=0.5, label="목표 스윙 피크")
    ax.set_title(leg_labels[i])
    ax.set_ylabel("상대 높이 (m)")
    ax.grid(True, alpha=0.3)
axs[1, 0].set_xlabel("시간 (s)")
axs[1, 1].set_xlabel("시간 (s)")
plt.suptitle("Unitree Go1 네 발의 스윙 높이 궤적 (Swing Peak 2x2 Grid)", y=1.02, fontweight="bold")
plt.tight_layout()
plt.show()

# 선속도 추종 및 외란(t=2.2s) 반응 플롯
fig, axes = plt.subplots(2, 1, figsize=(10, 4.2), sharex=True)
vx_smooth = np.convolve(linvel_arr[:, 0], np.ones(5) / 5, mode="same")
vy_smooth = np.convolve(linvel_arr[:, 1], np.ones(5) / 5, mode="same")

axes[0].plot(time_axis, vx_smooth, color="royalblue", linewidth=2, label="실제 전진 속도 vx")
axes[0].plot(time_axis, cmd_arr[:, 0], "r--", linewidth=1.8, label="명령 전진 속도 cmd_vx")
axes[0].set_ylabel("vx (m/s)")
axes[0].set_title("전진 속도 추종 및 측면 외란(Velocity Kick at t=2.2s) 회복")
axes[0].legend(loc="upper left")
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_axis, vy_smooth, color="darkorange", linewidth=2, label="실제 측면 속도 vy")
axes[1].plot(time_axis, cmd_arr[:, 1], "r--", linewidth=1.8, label="명령 측면 속도 cmd_vy")
axes[1].axvline(2.2, color="purple", linestyle=":", linewidth=1.8, label="측면 충격 인가 (t=2.2s)")
axes[1].set_ylabel("vy (m/s)")
axes[1].set_xlabel("시간 (s)")
axes[1].legend(loc="upper left")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Sim-to-Real을 위한 도메인 랜덤화 (Domain Randomization)

실제 로봇에 강화학습 정책을 배포할 때 가장 큰 문제는 시뮬레이터와 현실 세계 간의 물리적 차이(**Sim-to-Real Gap**)입니다.
`mujoco_playground`의 `locomotion/go1/randomize.py`에서는 `mjx.Model`의 배열 필드를 `jax.vmap`으로 복제하여 각 병렬 환경마다 **바닥 마찰계수(`geom_friction`)**, **관절 전기자 관성(`dof_armature`)**, **링크 질량(`body_mass`)**을 무작위화합니다. 아래 코드로 이 패턴을 직접 확인합니다.

In [ ]:
# 4. Playground 스타일 MJX 도메인 랜덤화(Domain Randomization) 함수 구현
def domain_randomize_model(mj_model: mujoco.MjModel, num_worlds: int = 8, seed: int = 0) -> Dict[str, jax.Array]:
    """
    num_worlds 개의 병렬 환경을 위해 바닥 마찰계수, 관절 아마추어, 몸통 질량 오프셋을 무작위 샘플링합니다.
    (mujoco_playground._src.locomotion.go1.randomize 패턴)
    """
    rng = jax.random.PRNGKey(seed)

    @jax.vmap
    def sample_world_params(key):
        k1, k2, k3 = jax.random.split(key, 3)
        # 1) 지면 마찰계수: U(0.4, 1.4)
        floor_friction = jax.random.uniform(k1, (), minval=0.4, maxval=1.4)
        # 2) 관절 전기자 관성(armature) 배율: U(1.0, 1.08)
        armature_scale = jax.random.uniform(k2, (mj_model.nv,), minval=1.0, maxval=1.08)
        # 3) 몸통(trunk) 추가 하중(Payload): U(-0.8kg, +1.5kg)
        payload_mass = jax.random.uniform(k3, (), minval=-0.8, maxval=1.5)
        return {
            "floor_friction": floor_friction,
            "dof_armature": jp.array(mj_model.dof_armature) * armature_scale,
            "trunk_mass": jp.array(mj_model.body_mass[trunk_id]) + payload_mass,
        }

    keys = jax.random.split(rng, num_worlds)
    return sample_world_params(keys)

rand_batch = domain_randomize_model(go1_model, num_worlds=6)
print("=== 6개 병렬 월드의 무작위화된 물리 파라미터 ===")
print(" - 월드별 바닥 마찰계수 :", np.array(rand_batch["floor_friction"]))
print(" - 월드별 몸통(Trunk) 질량 (kg):", np.array(rand_batch["trunk_mass"]))

---
## 파트 4. 다지 로봇 손 조작(Dexterous Manipulation — LEAP Hand $SO(3)$ 큐브 회전) & 비대칭 액터-크리틱

`mujoco_playground`의 `manipulation.ipynb`에서는 매니퓰레이터 팔(`PandaPickCube`)에서 나아가, 16자유도 다지 로봇 손(Multi-fingered Dexterous Hand)인 **LEAP Hand**가 손바닥 위의 큐브를 떨어뜨리지 않고 목표 $SO(3)$ 3차원 자세로 회전시키는 **`LeapCubeReorient`** 환경을 다룹니다.

### 핵심 기법: 비대칭 액터-크리틱 (Asymmetric Actor-Critic)
실제 로봇 손에는 물체 내부의 접촉력이나 정확한 마찰계수를 측정할 센서가 없습니다. 따라서 Playground는 관측값을 딕셔너리(`Dict[str, Array]`)로 구성합니다:
- **`obs["state"]` (Actor 입력)**: 실제 하드웨어에서도 얻을 수 있는 손가락 관절 위치(`qpos`), 속도(`qvel`), 이전 행동(`last_act`), 카메라로 추정한 큐브 위치·쿼터니언 오차
- **`obs["privileged_state"]` (Critic 입력)**: `state`에 더해 시뮬레이터 내부에서만 정확히 알 수 있는 **큐브 선속도/각속도, 손가락 끝(Fingertip) 3D 위치, 액추에이터 반력(`actuator_force`)**을 포함하여 가치 함수(Value Function) 학습 속도를 극대화합니다.

In [ ]:
# 1. Menagerie의 LEAP Hand (우측 손) 모델에 조작 대상 3D 컬러 큐브를 결합한 인핸드 조작 환경 생성
leap_right_xml = MENAGERIE_DIR / "leap_hand" / "right_hand.xml"

LEAP_CUBE_SCENE_XML = f"""
<mujoco model="leap_cube_reorient">
  <include file="{leap_right_xml.as_posix()}"/>
  <statistic center="0.0 0.0 0.08" extent="0.28"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1"/>
    <global azimuth="140" elevation="-35"/>
  </visual>
  <worldbody>
    <light pos="0 0 0.6" dir="0 0 -1" directional="true"/>
    <camera name="closeup" pos="0.22 -0.18 0.22" xyaxes="0.64 0.77 0 -0.45 0.38 0.81"/>
    <!-- 손바닥 위에 놓인 조작 대상 3D 큐브 -->
    <body name="cube" pos="0.055 -0.025 0.095">
      <freejoint name="cube_freejoint"/>
      <geom name="cube_box" type="box" size="0.026 0.026 0.026" mass="0.065"
            friction="1.2 0.01 0.002" rgba="0.92 0.28 0.2 1"/>
      <geom name="marker_x" type="cylinder" fromto="0 0 0 0.028 0 0" size="0.005" rgba="1 0.9 0.1 1" contype="0" conaffinity="0"/>
      <geom name="marker_z" type="cylinder" fromto="0 0 0 0 0 0.028" size="0.005" rgba="0.1 0.7 1 1" contype="0" conaffinity="0"/>
    </body>
    <!-- 목표 자세(Goal Orientation) 시각화용 반투명 레퍼런스 큐브 -->
    <body name="goal_cube" mocap="true" pos="0.055 0.11 0.12" quat="0.8535534 0.3535534 0.3535534 0.1464466">
      <geom type="box" size="0.022 0.022 0.022" rgba="0.2 0.85 0.4 0.45" contype="0" conaffinity="0"/>
      <geom type="cylinder" fromto="0 0 0 0.025 0 0" size="0.004" rgba="1 0.9 0.1 0.7" contype="0" conaffinity="0"/>
      <geom type="cylinder" fromto="0 0 0 0 0 0.025" size="0.004" rgba="0.1 0.7 1 0.7" contype="0" conaffinity="0"/>
    </body>
  </worldbody>
</mujoco>
"""

leap_model = mujoco.MjModel.from_xml_string(LEAP_CUBE_SCENE_XML)
leap_data = mujoco.MjData(leap_model)
mujoco.mj_forward(leap_model, leap_data)

print(f"LEAP Hand + Cube 모델 로드 완료! (총 qpos 차원: {leap_model.nq}, 손가락 액추에이터 수: {leap_model.nu})")

In [ ]:
# 2. 비대칭 액터-크리틱(Asymmetric Actor-Critic) 관측 추출 함수 및 인핸드 큐브 파지·회전 시뮬레이션
def get_asymmetric_observations(model: mujoco.MjModel, data: mujoco.MjData, goal_quat: np.ndarray) -> Dict[str, np.ndarray]:
    """
    Playground의 LeapCubeReorient 환경처럼 Actor용 'state'와 Critic용 'privileged_state'를 분리하여 반환합니다.
    """
    hand_qpos = data.qpos[:16]
    hand_qvel = data.qvel[:16]
    cube_pos = data.qpos[16:19]
    cube_quat = data.qpos[19:23]
    cube_linvel = data.qvel[16:19]
    cube_angvel = data.qvel[19:22]

    # 쿼터니언 상대 자세 오차 계산
    quat_diff = np.zeros(4)
    q_inv = np.array([goal_quat[0], -goal_quat[1], -goal_quat[2], -goal_quat[3]])
    mujoco.mju_mulQuat(quat_diff, cube_quat, q_inv)

    # Actor 관측 (실제 로봇 센서로 관측 가능한 39차원 벡터)
    actor_state = np.concatenate([hand_qpos, hand_qvel * 0.1, cube_pos, quat_diff])

    # Critic 특권 관측 (시뮬레이터 내부 동역학/접촉/토크 정보까지 포함한 61차원 벡터)
    privileged_state = np.concatenate([
        actor_state,
        cube_linvel,
        cube_angvel,
        data.actuator_force * 0.1,
    ])
    return {"state": actor_state, "privileged_state": privileged_state}

def quat_geodesic_angle(q1: np.ndarray, q2: np.ndarray) -> float:
    """SO(3) 상에서 두 단위 쿼터니언 사이의 회전 각도 오차(rad)를 반환합니다."""
    dot = np.clip(np.abs(np.dot(q1, q2)), -1.0, 1.0)
    return float(2.0 * np.arccos(dot))

# 시뮬레이션 및 렌더링 실행
mujoco.mj_resetData(leap_model, leap_data)
# 초기 손바닥 오목 파지 자세 설정
base_grasp = np.zeros(leap_model.nu)
base_grasp[[1, 5, 9]] = 0.55      # 검지/중지/약지 MCP 살짝 굽힘
base_grasp[[2, 6, 10]] = 0.65     # PIP 굽힘
base_grasp[[3, 7, 11]] = 0.45     # DIP 굽힘
base_grasp[[12, 13, 14, 15]] = [0.6, 0.4, 0.5, 0.5]  # 엄지 대향 자세
leap_data.qpos[:16] = base_grasp * 0.6
leap_data.ctrl[:] = base_grasp
mujoco.mj_forward(leap_model, leap_data)

goal_quat = leap_data.mocap_quat[0].copy()
obs_sample = get_asymmetric_observations(leap_model, leap_data, goal_quat)
print(f"Actor 관측('state') 차원: {obs_sample['state'].shape} | Critic 특권 관측('privileged_state') 차원: {obs_sample['privileged_state'].shape}")

leap_renderer = mujoco.Renderer(leap_model, height=360, width=640)
leap_frames = []
angle_errors = []

for step in range(450):
    t = step * leap_model.opt.timestep
    # 손가락 협응 진동으로 손바닥 위 큐브를 지지하며 회전 유도
    ctrl = base_grasp.copy()
    ctrl[0] = 0.25 * np.sin(2.5 * t)          # 검지 좌우 롤링
    ctrl[4] = 0.25 * np.sin(2.5 * t + 1.0)    # 중지 좌우 롤링
    ctrl[8] = 0.25 * np.sin(2.5 * t + 2.0)    # 약지 좌우 롤링
    ctrl[13] = 0.4 + 0.25 * np.cos(2.5 * t)   # 엄지 회전 토크 인가

    leap_data.ctrl[:] = ctrl
    mujoco.mj_step(leap_model, leap_data)

    if step % 3 == 0:
        cube_quat = leap_data.qpos[19:23]
        angle_errors.append(quat_geodesic_angle(cube_quat, goal_quat))
        leap_renderer.update_scene(leap_data, camera="closeup")
        leap_frames.append(leap_renderer.render())

leap_renderer.close()
print("빨간색 큐브: 조작 대상 물체 | 초록색 반투명 큐브: 목표 SO(3) 회전 자세")
media.show_video(leap_frames, fps=50)

---
## 파트 5. `mujoco_playground` & `brax` PPO 강화학습 및 커리큘럼 파인튜닝 템플릿

`uv sync --extra playground` (또는 `pip install playground brax`)로 `mujoco_playground` 패키지를 설치한 경우, 아래 헬퍼 함수를 통해 **별도로 `mujoco_menagerie`를 중복 다운로드하지 않고** 우리 저장소의 `assets/mujoco_menagerie` 서브모듈을 바로 연결하여 PPO/SAC 학습과 체크포인트 파인튜닝을 수행할 수 있습니다.

In [ ]:
def configure_playground_with_local_menagerie():
    """
    mujoco_playground가 외부에서 mujoco_menagerie를 다시 클론하지 않도록
    현재 프로젝트의 assets/mujoco_menagerie 서브모듈 경로를 직접 바인딩합니다.
    """
    try:
        from etils import epath
        from mujoco_playground._src import mjx_env
        mjx_env.MENAGERIE_PATH = epath.Path(MENAGERIE_DIR)
        print(f"[연결 완료] mujoco_playground가 로컬 서브모듈을 사용합니다: {mjx_env.MENAGERIE_PATH}")
        return True
    except ImportError:
        print("[안내] mujoco_playground 패키지가 설치되어 있지 않습니다. (선택 설치: uv pip install playground brax)")
        return False

has_playground = configure_playground_with_local_menagerie()

# mujoco_playground가 설치된 환경에서의 Brax PPO 학습 & 에너지 페널티 파인튜닝 실행 예시
if has_playground:
    from mujoco_playground import registry, wrapper
    from mujoco_playground.config import dm_control_suite_params, locomotion_params, manipulation_params

    print("DM Control Suite 환경 목록 (일부):", list(registry.dm_control_suite.ALL_ENVS)[:6])
    print("Locomotion 환경 목록 (일부):", list(registry.locomotion.ALL_ENVS)[:6])
    print("Manipulation 환경 목록 (일부):", list(registry.manipulation.ALL_ENVS)[:6])

### Sim-to-Real 에너지·토크 페널티 커리큘럼 파인튜닝 (Handstand / Locomotion 예시)

`mujoco_playground` 기술 보고서에서 실제 로봇(Go1 물구나무서기, LEAP Hand 큐브 회전 등)에 정책을 배포할 때 사용하는 핵심 레시피는 **2단계 커리큘럼 파인튜닝**입니다:

1. **1단계 (탐색 중심 초기 학습)**: 에너지 페널티(`reward_config.energy = 0.0`)를 끄고 높은 최대 토크를 허용하여 에이전트가 목표 동작(예: 물구나무서기, 큐브 회전)을 빠르게 발견하도록 학습하고 Orbax 체크포인트로 저장합니다.
2. **2단계 (Sim-to-Real 스무딩 파인튜닝)**: 저장된 체크포인트(`restore_checkpoint_path`)에서 이어서 학습하되, 에너지 소비 및 관절 가속도 페널티를 활성화하여 고주파 떨림(Jitter)을 제거합니다:

```python
# 2단계 Sim-to-Real 스무딩을 위한 보상 설정 오버라이드 예시
env_cfg = registry.get_default_config("Go1Handstand")
env_cfg.energy_termination_threshold = 400   # 과도한 전력 소모 시 조기 종료 임계값 하향
env_cfg.reward_config.energy = -0.003        # 토크 × 각속도(|τ · q̇|) 에너지 페널티 부여
env_cfg.reward_config.dof_acc = -2.5e-7      # 급격한 관절 가속도(q̈) 페널티 부여
```